# mini-vLLM — GPU Benchmark & Correctness Notebook

**Runtime:** GPU (T4) — Kaggle: Settings → Accelerator → GPU T4 x1

Self-contained: clone, install, run. No local setup needed.

In [ ]:
# ── Setup (run once) ──────────────────────────────────────────────────────
import subprocess, sys

# Clone repo (skip if already present)
result = subprocess.run(['git', 'clone', 'https://github.com/shlokkvaishnav/LLM-Inference-Engine.git', 'mini-vllm'],
                        capture_output=True, text=True)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print(result.stderr)
else:
    print('Repo ready.')

%cd mini-vllm
!git pull  # get latest commits

# Install with GPU extras (triton for M4+)
!pip install -e '.[gpu]' -q

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## M1 + M2 — Correctness tests

Run the full test suite. Both tests must pass before any GPU benchmarking.

- **M1:** our decode loop matches `model.generate()` token-for-token (greedy)
- **M2:** each sequence in a static batch matches its single-sequence result

In [ ]:
# Run on GPU with TinyLlama (set MODEL env var to override)
import os
os.environ.setdefault('MINI_VLLM_TEST_MODEL', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0')
os.environ.setdefault('MINI_VLLM_TEST_DEVICE', 'cuda')

!python -m pytest tests/test_correctness.py -v -s 2>&1 | head -80

## M3 — Continuous batching (added when milestone lands)

## M4 — Paged KV-cache (added when milestone lands)

## M5 — Quantization (added when milestone lands)

## M7 — Benchmark suite + load test (added when milestone lands)